# 01 - Extract (Bronze): SQL Server → Partitioned Parquet

**Purpose:**  
Connect to SQL Server via JDBC, load the cleaned view `vw_big_five_clean`, and persist the data as partitioned Parquet files (bronze layer) for downstream processing.

---

## When to run
- Initially, to bootstrap the dataset
- Re-run when new data appears in SQL Server
- Can be scheduled via Airflow DAG for periodic ingestion

---

## Output
```
data/bronze/big_five/
  ├── year_month=2016-03/
  │     └── part-*.parquet
  ├── year_month=2016-04/
  │     └── part-*.parquet
  ├── ...
  └── _bronze_meta.json
```

- **Format:** Parquet (Snappy compressed)
- **Mode:** overwrite (full refresh - for incremental, switch to append + dedup)
- **Partitioning:** `year_month` - enables partition pruning in downstream queries

---

## Pipeline position
```
[SQL Server: vw_big_five_clean]  ← this notebook
         ↓
[Bronze: data/bronze/big_five/]  ← raw, partitioned, immutable
         ↓
[02_eda.ipynb]                   ← exploratory analysis
         ↓
[03_transform_silver.ipynb]      ← reverse scoring, trait scores
         ↓
[04_aggregate_gold.ipynb]        ← analytics-ready aggregates
         ↓
[05_visualizations.ipynb]        ← Plotly charts
```

---

## Bronze layer principles
- **No transformations** - data lands exactly as it comes from the source
- **Immutable** - never modify bronze; reprocess by re-running this notebook
- **Traceable** - every ingestion writes metadata (row count, timestamp, checksum)

---

## Prerequisites
- SQL Server 2019 running on `localhost:1433` (instance: `SQLEXPRESS`)
- Database `BigFiveDB` with view `vw_big_five_clean` (see `sql/02_create_view.sql`)
- JDBC driver `.jar` in `drivers/`
- `mssql-jdbc_auth-13.2.1.x64.dll` in `C:/Windows/System32/` (Windows Authentication)

## Setup & Configuration

In [1]:
import os
import json
from datetime import datetime, timezone

import pandas as pd
from pyspark.sql import SparkSession, functions as F

# Display settings for pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)

# Paths
JDBC_JAR = os.path.abspath('../drivers/mssql-jdbc-13.2.1.jre11.jar')
BRONZE_PATH = '../data/bronze/big_five'
META_PATH = f"{BRONZE_PATH}/_bronze_meta.json"

# Source
JDBC_URL = (
    'jdbc:sqlserver://localhost:1433;'
    'databaseName=BigFiveDB;'
    'integratedSecurity=true;'
    'trustServerCertificate=true;'
)

JDBC_DRIVER = 'com.microsoft.sqlserver.jdbc.SQLServerDriver'
SOURCE_VIEW  = 'vw_big_five_clean'

# Expected schema constants
EXPECTED_COLS = 60
EXPECTED_MIN_ROWS = 500_000   # sanity lower bound
SCORE_MIN, SCORE_MAX = 1.0, 5.0

INGESTION_TS = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

print(f"JDBC jar found: {os.path.exists(JDBC_JAR)}")
print(f"Bronze output: {os.path.abspath(BRONZE_PATH)}")
print(f"Ingestion ts: {INGESTION_TS}")

JDBC jar found: True
Bronze output: C:\Users\micha\PycharmProjects\pipeline-big_five_personality\data\bronze\big_five
Ingestion ts: 2026-03-18T17:57:39Z


## Spark Session

In [2]:
spark = SparkSession.builder \
    .appName('BigFive_Extract') \
    .master('local[*]') \
    .config('spark.jars', JDBC_JAR) \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f"Spark version : {spark.version}")

Spark version : 4.1.1


## Extract from SQL Server

We read from `vw_big_five_clean` - a SQL view that has already:
- Filtered invalid item responses (outside 1–5 range)
- Removed bots (`IPC > 1`) and abandoned sessions (`testelapse` outside 60–3600s)
- Pre-aggregated response times per Big Five trait

No further transformation happens here. Bronze = raw source data.

In [3]:
print(f"Reading from: {SOURCE_VIEW} ...")

df_bronze = (
    spark.read
    .format('jdbc')
    .option('url', JDBC_URL)
    .option('dbtable', SOURCE_VIEW)
    .option('driver', JDBC_DRIVER)
    .load()
    .withColumn('ingestion_ts', F.lit(INGESTION_TS))
    .withColumn('year_month', F.date_format(F.col('test_date'), 'yyyy-MM'))
)

row_count = df_bronze.count()
col_count = len(df_bronze.columns)

print(f"Rows: {row_count:,}")
print(f"Columns: {col_count} (60 source + ingestion_ts + year_month)")

Reading from: vw_big_five_clean ...
Rows: 597,146
Columns: 62 (60 source + ingestion_ts + year_month)


## Schema Inspection

In [4]:
df_bronze.printSchema()

root
 |-- EXT1: double (nullable = true)
 |-- EXT2: double (nullable = true)
 |-- EXT3: double (nullable = true)
 |-- EXT4: double (nullable = true)
 |-- EXT5: double (nullable = true)
 |-- EXT6: double (nullable = true)
 |-- EXT7: double (nullable = true)
 |-- EXT8: double (nullable = true)
 |-- EXT9: double (nullable = true)
 |-- EXT10: double (nullable = true)
 |-- EST1: double (nullable = true)
 |-- EST2: double (nullable = true)
 |-- EST3: double (nullable = true)
 |-- EST4: double (nullable = true)
 |-- EST5: double (nullable = true)
 |-- EST6: double (nullable = true)
 |-- EST7: double (nullable = true)
 |-- EST8: double (nullable = true)
 |-- EST9: double (nullable = true)
 |-- EST10: double (nullable = true)
 |-- AGR1: double (nullable = true)
 |-- AGR2: double (nullable = true)
 |-- AGR3: double (nullable = true)
 |-- AGR4: double (nullable = true)
 |-- AGR5: double (nullable = true)
 |-- AGR6: double (nullable = true)
 |-- AGR7: double (nullable = true)
 |-- AGR8: double (nu

## Data Preview

Using `.toPandas()` on a small sample for a readable table view.

> **Warning:** Never call `.toPandas()` on the full dataset - it loads everything into RAM. Always use `.limit(N)` for previews.

In [5]:
df_bronze.limit(5).toPandas()

,EXT1,EXT2,EXT3,EXT4,EXT5,EXT6,EXT7,EXT8,EXT9,EXT10,EST1,EST2,EST3,EST4,EST5,EST6,EST7,EST8,EST9,EST10,AGR1,AGR2,AGR3,AGR4,AGR5,AGR6,AGR7,AGR8,AGR9,AGR10,CSN1,CSN2,CSN3,CSN4,CSN5,CSN6,CSN7,CSN8,CSN9,CSN10,OPN1,OPN2,OPN3,OPN4,OPN5,OPN6,OPN7,OPN8,OPN9,OPN10,testelapse,country,lat,lon,test_date,ext_response_ms,est_response_ms,agr_response_ms,csn_response_ms,opn_response_ms,ingestion_ts,year_month
0,4.0,1.0,5.0,2.0,5.0,1.0,5.0,2.0,4.0,1.0,1.0,4.0,4.0,2.0,2.0,2.0,2.0,2.0,3.0,2.0,2.0,5.0,2.0,4.0,2.0,3.0,2.0,4.0,3.0,4.0,3.0,4.0,3.0,2.0,2.0,4.0,4.0,2.0,4.0,4.0,5.0,1.0,4.0,1.0,4.0,1.0,5.0,3.0,4.0,5.0,234.0,GB,51.5448,0.1991,2016-03-03,46568.0,47699.0,52419.0,49929.0,35571.0,2026-03-18T17:57:39Z,2016-03
1,3.0,5.0,3.0,4.0,3.0,3.0,2.0,5.0,1.0,5.0,2.0,3.0,4.0,1.0,3.0,1.0,2.0,1.0,3.0,1.0,1.0,4.0,1.0,5.0,1.0,5.0,3.0,4.0,5.0,3.0,3.0,2.0,5.0,3.0,3.0,1.0,3.0,3.0,5.0,3.0,1.0,2.0,4.0,2.0,3.0,1.0,4.0,2.0,5.0,3.0,179.0,MY,3.1698,101.706,2016-03-03,37527.0,34479.0,29662.0,42629.0,32604.0,2026-03-18T17:57:39Z,2016-03
2,2.0,3.0,4.0,4.0,3.0,2.0,1.0,3.0,2.0,5.0,4.0,4.0,4.0,2.0,2.0,2.0,2.0,2.0,1.0,3.0,1.0,4.0,1.0,4.0,2.0,4.0,1.0,4.0,4.0,3.0,4.0,2.0,2.0,2.0,3.0,3.0,4.0,2.0,4.0,2.0,5.0,1.0,2.0,1.0,4.0,2.0,5.0,3.0,4.0,4.0,186.0,GB,54.9119,-1.3833,2016-03-03,47347.0,35996.0,36975.0,34700.0,29522.0,2026-03-18T17:57:39Z,2016-03
3,2.0,2.0,2.0,3.0,4.0,2.0,2.0,4.0,1.0,4.0,3.0,3.0,3.0,2.0,3.0,2.0,2.0,2.0,4.0,3.0,2.0,4.0,3.0,4.0,2.0,4.0,2.0,4.0,3.0,4.0,2.0,4.0,4.0,4.0,1.0,2.0,2.0,3.0,1.0,4.0,4.0,2.0,5.0,2.0,3.0,1.0,4.0,4.0,3.0,3.0,219.0,GB,51.75,-1.25,2016-03-03,44948.0,40478.0,52016.0,43177.0,37643.0,2026-03-18T17:57:39Z,2016-03
4,3.0,3.0,4.0,2.0,4.0,2.0,2.0,3.0,3.0,4.0,3.0,4.0,3.0,2.0,2.0,1.0,2.0,1.0,2.0,2.0,2.0,3.0,1.0,4.0,2.0,3.0,2.0,3.0,4.0,4.0,3.0,2.0,4.0,1.0,3.0,2.0,4.0,3.0,4.0,3.0,5.0,1.0,5.0,1.0,3.0,1.0,5.0,4.0,5.0,2.0,196.0,SE,59.3333,18.05,2016-03-03,44137.0,29509.0,37211.0,50927.0,30593.0,2026-03-18T17:57:39Z,2016-03


## Quality Gate

Lightweight checks **before** writing. If any assertion fails, the write is blocked.

| Check | Expected |
|---|---|
| Row count | ≥ 500,000 |
| Column count | 62 (60 source + ingestion_ts + year_month) |
| Null count in key cols | 0 |
| Item score range | [1.0, 5.0] across all 50 items |

In [6]:
item_cols = (
    [f"EXT{i}" for i in range(1, 11)] +
    [f"EST{i}" for i in range(1, 11)] +
    [f"AGR{i}" for i in range(1, 11)] +
    [f"CSN{i}" for i in range(1, 11)] +
    [f"OPN{i}" for i in range(1, 11)]
)
key_meta_cols = ['country', 'test_date', 'testelapse']

checks_passed = True

# Row count
if row_count >= EXPECTED_MIN_ROWS:
    print(f"Row count: {row_count:,} (≥ {EXPECTED_MIN_ROWS:,})")
else:
    print(f"Row count: {row_count:,} - BELOW expected minimum!")
    checks_passed = False

# Column count
expected_cols_with_ts = EXPECTED_COLS + 2  # +ingestion_ts +year_month
if col_count == expected_cols_with_ts:
    print(f"Column count: {col_count} (as expected)")
else:
    print(f"Column count: {col_count} - expected {expected_cols_with_ts}!")
    checks_passed = False

# Nulls in key columns
null_counts = df_bronze.select(
    [F.sum(F.col(c).isNull().cast('int')).alias(c) for c in key_meta_cols]
).toPandas()
total_nulls = null_counts.values.sum()
if total_nulls == 0:
    print(f"Nulls in key cols: 0")
else:
    print(f"Nulls found in key columns:")
    print(null_counts.T)
    checks_passed = False

# Score range across all 50 items
range_check = df_bronze.select(
    F.min(F.least(*[F.col(c) for c in item_cols])).alias('global_min'),
    F.max(F.greatest(*[F.col(c) for c in item_cols])).alias('global_max')
).collect()[0]

if range_check['global_min'] >= SCORE_MIN and range_check['global_max'] <= SCORE_MAX:
    print(f"Score range: [{range_check['global_min']}, {range_check['global_max']}] - within [1.0, 5.0]")
else:
    print(f"Score range out of bounds: [{range_check['global_min']}, {range_check['global_max']}]")
    checks_passed = False

# Gate
print()
if not checks_passed:
    raise ValueError('Quality gate FAILED -> aborting write. Fix issues above before proceeding.')
else:
    print('All quality checks passed -> proceeding to write.')

Row count: 597,146 (≥ 500,000)
Column count: 62 (as expected)
Nulls in key cols: 0
Score range: [1.0, 5.0] - within [1.0, 5.0]

All quality checks passed -> proceeding to write.


## Write Bronze Layer

We partition by `year_month` which gives downstream notebooks the ability to read only a specific date range without scanning the whole dataset (partition pruning).

Example downstream read:
```python
# Read only 2016 data - Spark skips all other partitions
df = spark.read.parquet(BRONZE_PATH).filter(F.col('year_month') >= '2016-01')
```

In [7]:
print(f"Writing to: {BRONZE_PATH}")
print(f"Partitioned by: year_month")

df_bronze.write.mode('overwrite').partitionBy('year_month').parquet(BRONZE_PATH)

print('Write complete.')

Writing to: ../data/bronze/big_five
Partitioned by: year_month
Write complete.


## Write Ingestion Metadata

Every ingestion writes a `_bronze_meta.json` file alongside the data.

This enables monitoring, auditing, and Airflow sensors to detect fresh data.

In [8]:
# Collect date range from the actual data
date_range = df_bronze.select(
    F.min('test_date').alias('min_date'),
    F.max('test_date').alias('max_date')
).collect()[0]

# Count unique dates (= number of partitions written)
n_partitions = df_bronze.select('year_month').distinct().count()

meta = {
    'ingestion_ts': INGESTION_TS,
    'source_view': SOURCE_VIEW,
    'source_db': 'BigFiveDB',
    'rows_written': row_count,
    'columns': col_count,
    'partition_col': 'year_month',
    'n_partitions': n_partitions,
    'date_range_min': str(date_range['min_date']),
    'date_range_max': str(date_range['max_date']),
    'score_range': {'min': range_check['global_min'], 'max': range_check['global_max']},
    'output_path': os.path.abspath(BRONZE_PATH),
    'compression': 'snappy',
    'quality_gate': 'passed',
    'spark_version': spark.version,
}

os.makedirs(BRONZE_PATH, exist_ok=True)
with open(META_PATH, 'w') as f:
    json.dump(meta, f, indent=2)

print(f"Metadata written to: {META_PATH}")
print(json.dumps(meta, indent=2))

Metadata written to: ../data/bronze/big_five/_bronze_meta.json
{
  "ingestion_ts": "2026-03-18T17:57:39Z",
  "source_view": "vw_big_five_clean",
  "source_db": "BigFiveDB",
  "rows_written": 597146,
  "columns": 62,
  "partition_col": "year_month",
  "n_partitions": 33,
  "date_range_min": "2016-03-03",
  "date_range_max": "2018-11-08",
  "score_range": {
    "min": 1.0,
    "max": 5.0
  },
  "output_path": "C:\\Users\\micha\\PycharmProjects\\pipeline-big_five_personality\\data\\bronze\\big_five",
  "compression": "snappy",
  "quality_gate": "passed",
  "spark_version": "4.1.1"
}


## Verify Write

Read back the bronze layer and assert row and column counts match the source.

In [11]:
df_verify = spark.read.parquet(BRONZE_PATH)

verified_rows = df_verify.count()
verified_cols = len(df_verify.columns)

assert verified_rows == row_count, \
    f"Row count mismatch! Written: {row_count:,}, Read back: {verified_rows:,}"
assert verified_cols == col_count, \
    f"Column count mismatch! Written: {col_count}, Read back: {verified_cols}"

# Show partition structure
date_partition_counts = (
    df_verify
    .groupBy('year_month')
    .count()
    .orderBy('year_month')
    .toPandas()
)

print('Verification passed!')
print(f"Rows verified: {verified_rows:,}")
print(f"Columns verified: {verified_cols}")
print(f"Partitions: {n_partitions} unique months")
print(f"Date range: {date_range['min_date']} → {date_range['max_date']}")
print('\nSample of partition sizes (rows per year_month):')
print(date_partition_counts.head(10).to_string(index=False))

Verification passed!
Rows verified: 597,146
Columns verified: 62
Partitions: 33 unique months
Date range: 2016-03-03 → 2018-11-08

Sample of partition sizes (rows per year_month):
year_month  count
   2016-03  21655
   2016-04  23666
   2016-05  26235
   2016-06  18582
   2016-07  15953
   2016-08  15896
   2016-09  15893
   2016-10  18838
   2016-11  13894
   2016-12  17354


## Example: Partition Pruning

Demonstrating that downstream notebooks can efficiently read a subset of data without scanning all partitions.

In [12]:
# Read only data from 2016 - Spark skips all 2017/2018 partitions
df_2016 = spark.read.parquet(BRONZE_PATH).filter(F.col('year_month') < '2017-01')

rows_2016 = df_2016.count()
pct_2016 = rows_2016 / row_count * 100

print(f"2016 respondents : {rows_2016:,} ({pct_2016:.1f}% of total)")
print('Partition pruning works -> only 2016 partitions were scanned.')

2016 respondents : 187,966 (31.5% of total)
Partition pruning works -> only 2016 partitions were scanned.


## Summary

| Step | Details |
|---|---|
| **Source** | SQL Server 2019 - `BigFiveDB.vw_big_five_clean` |
| **Connection** | JDBC, Windows Authentication |
| **Rows extracted** | ~597,000 |
| **Columns** | 62 (60 source + `ingestion_ts` + `year_month`) |
| **Output** | `data/bronze/big_five/` - partitioned by `year_month` |
| **Compression** | Snappy |
| **Quality gate** | Passed |
| **Metadata** | `data/bronze/big_five/_bronze_meta.json` |
| **Next notebook** | `02_eda.ipynb` |